# E6: the transformation battery, reconditioned

The published table ranks semantics-preserving rewrites by how far macro-F1
falls when each is applied to a whole evaluation set. Reviewer S9 pointed out
that this is confounded, and it is: the rewrites fire at very different rates.

| Rewrite | Applied to |
|---|---|
| rename identifiers | 100.0% |
| normalise whitespace | 85.3% |
| strip comments | 81.0% |
| insert dead code | 100.0% |
| compress blank lines | 5.9% |

A rewrite that fires on more files moves the aggregate more whatever it does
per file, so the claim that renaming costs the most was never actually
established against stripping comments.

## What this run fixes

**The conditional effect, computed rather than derived.** The suite now scores
the baseline and the rewritten inputs on **the same altered rows**. The plan
suggested dividing the aggregate delta by the application rate; that is invalid,
because macro-F1 is an average of per-class ratios and does not decompose as a
weighted sum over rows. On a synthetic case with a constant true effect the
quotient errs by up to 0.216, and by 0.093 at exactly the 81% rate strip
comments has. The counterexample is pinned as a test.

**n = 10,000 rather than 2,000**, on a **named** condition, **stratified by
class**. All three were part of the same finding.

What this run does not yet cover, and the paper should not claim: parse-success
validation of the rewrites, AST-level structural rewrites, and running the
battery across the whole detector panel rather than the published detector
alone.

## Settings

| Setting | Value |
|---|---|
| **Accelerator** | `GPU T4 x2` |
| **Internet** | `On` |
| **Persistence** | `Files only` |

**Save Version -> Save & Run All (Commit).** Budget about 1.5 h: inference
only, six passes over 10,000 rows.


In [ ]:
import os, sys, time, subprocess, shutil, pathlib, json

T0 = time.time()
def elapsed(label=""):
    m = (time.time() - T0) / 60
    print(f"[{m:6.1f} min] {label}", flush=True)

def run(cmd):
    """Run a stage and stop the notebook if it fails, rather than letting the
    next cell train on whatever stale data is lying around."""
    print(">>", " ".join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, "-u", *[str(c) for c in cmd]])
    if r.returncode != 0:
        raise SystemExit(f"FAILED: {' '.join(str(c) for c in cmd)}")

print("Python", sys.version.split()[0])
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB")
else:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 in the right panel.")

## 1. Code

In [ ]:
WORK = pathlib.Path("/kaggle/working/project")
WORK.mkdir(parents=True, exist_ok=True)

def find_aicd():
    root = pathlib.Path("/kaggle/input")
    if not root.exists():
        return None
    for cand in root.rglob("aicd"):
        if (cand / "config.py").exists() and (cand / "models").is_dir():
            return cand
    return None

src = find_aicd()
if src is None:
    raise SystemExit(
        "aicd/ not found under /kaggle/input.\n"
        "Right panel -> Input -> Add Input -> Datasets, then add the dataset\n"
        "you created from aicd-code.zip.")

dest = WORK / "aicd"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(src, dest)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code ->", dest)
if not (dest / "data" / "exposure_arms.py").exists():
    raise SystemExit("This code dataset predates E1. Re-upload aicd-code.zip.")

## 2. Dependencies

In [ ]:
pkgs = ["xgboost", "tree-sitter", "tree-sitter-language-pack",
        "datasets", "shap", "pyyaml", "scikit-learn", "pyarrow", "datasketch"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import importlib
for m in ["xgboost", "sklearn", "transformers", "datasets", "yaml", "datasketch"]:
    mod = importlib.import_module(m)
    print(f"  ok  {m:14s} {getattr(mod, '__version__', '')}")
elapsed("deps")

## 3. Resume, if a previous run is attached

In [ ]:
# Resume support. A fresh Kaggle session starts with an empty working
# directory, so `--resume` alone finds nothing, prints "starting fresh" and
# silently retrains from epoch 0. That is how seven hours disappear without
# anyone noticing, so the checkpoint is restored explicitly here.
#
# The corpus is restored too, and that matters more than it looks. Rebuilding
# is deterministic given the seed but NOT across library versions: the same
# pipeline produced 417,645 rows earlier and 417,431 today, a parse-filter
# difference. Resuming a checkpoint onto a corpus it was not trained on would
# be quietly wrong, so if a previous split is available we use it and skip the
# rebuild entirely.
#
# Attach the previous version's output: Input -> Add Input -> Notebook Output.
# Set True when this notebook is pushed as a RESUME. A resume that silently
# finds nothing and trains from scratch is the expensive failure: it looks like
# a normal run and costs a full session. With this on, the notebook refuses.
REQUIRE_RESUME = False

import torch
art = WORK / "aicd" / "artifacts"
(art / "data").mkdir(parents=True, exist_ok=True)
CKPT = "branch_a_e6_ckpt.pt"

def newest(pattern):
    hits = sorted(pathlib.Path("/kaggle/input").rglob(pattern),
                  key=lambda q: q.stat().st_mtime, reverse=True)
    return hits[0] if hits else None

sp = newest("splits.parquet")
RESTORED_SPLITS = False
if sp is not None:
    shutil.copy(sp, art / "data" / "splits.parquet")
    import pandas as _pd
    _n = len(_pd.read_parquet(art / "data" / "splits.parquet", columns=["label"]))
    print(f"restored splits.parquet from {sp}  ({_n:,} rows)")
    RESTORED_SPLITS = True
else:
    print("no previous splits.parquet found; the corpus will be built fresh")

ck = newest(CKPT)
if ck is None:
    if REQUIRE_RESUME:
        raise SystemExit(
            f"This notebook was pushed as a RESUME but no {CKPT} was found "
            "under /kaggle/input. Training from scratch here would waste a "
            "whole session and look like success. Attach the previous "
            "version's output (Input -> Add Input -> Notebook Output) and "
            "re-run.")
    print(f"no {CKPT} under /kaggle/input -- this will train from epoch 0.")
    print("If you meant to resume, attach the previous version's output.")
else:
    shutil.copy(ck, art / CKPT)
    _c = torch.load(art / CKPT, map_location="cpu", weights_only=False)
    print(f"restored {CKPT} from {ck}")
    print(f"  holds epoch {_c['epoch']}, so training resumes at epoch {_c['epoch'] + 1}")
    if not RESTORED_SPLITS:
        raise SystemExit(
            "A checkpoint was restored but its corpus was not. Rebuilding may "
            "produce a different split than the one this checkpoint was "
            "trained on, which would make the resumed run unsound. Attach the "
            "previous output so splits.parquet comes with it.")


## 4. Build the corpus

In [ ]:
CFG = "kaggle.yaml"

# One train shard, not three. One shard plus dev and test is 493,850 raw rows
# which filter to the 417,645 of the original GPU build, with 196,854 of them
# training. Three shards is the matched-scale corpus and gives roughly 545,000
# training rows, which is a different experiment.
if RESTORED_SPLITS:
    print("corpus restored from the previous run; skipping the rebuild so the")
    print("resumed model continues on exactly the data it was trained on.")
    import pandas as pd
    _sp = pd.read_parquet(WORK / "aicd" / "artifacts" / "data" / "splits.parquet",
                          columns=["split"])
    rows = int((_sp["split"] == "train").sum())
    print(f"training rows: {rows:,}")
else:
    run(["-m", "aicd.data.download", "--config", CFG, "--train-shards", "1"])
    elapsed("downloaded 1 shard")

    for stage in ["normalize", "filter", "splits"]:
        run(["-m", f"aicd.data.{stage}", "--config", CFG])
        elapsed(stage)

    run(["-m", "pytest", "aicd/tests/", "-q"])
    elapsed("integrity tests passed")

    sp_report = json.load(open(WORK / "aicd" / "eval" / "reports" / "splits.json"))
    rows = sp_report["train"]["rows"]
    print(f"\ntraining rows: {rows:,}   (expected 196,854)")
    if abs(rows - 196854) > 5000:
        raise SystemExit(
            f"Got {rows:,} training rows, expected about 196,854. This notebook "
            "must reproduce the original GPU build exactly, or the arms are not "
            "comparable with the existing model.")

## 5. The battery, at n = 10,000

In [ ]:
run(["-m", "aicd.eval.transform_suite", "--config", CFG,
     "--model", "droiddetect", "--slice", "s1_in_distribution",
     "--n", "10000", "--batch-size", "32"])
elapsed("transformation battery")

## 6. Aggregate against conditional

In [ ]:
import json
rep = WORK / "aicd" / "eval" / "reports" / "transform_suite_droiddetect.json"
r = json.load(open(rep))
print(f"slice {r['slice']}, n = {r['n']:,}")
print(f"baseline macro-F1 {r['baseline']['macro_f1']:.4f}")
print()
print(f"{'rewrite':22s} {'applied':>8s} {'altered':>8s} {'aggregate':>10s} "
      f"{'conditional':>12s}")
print("-" * 66)
rows = sorted(r["transforms"], key=lambda x: x.get("delta_macro_f1_conditional") or 0)
for t in rows:
    cond = t.get("delta_macro_f1_conditional")
    cs = f"{cond:+12.4f}" if cond == cond and cond is not None else f"{'n/a':>12s}"
    print(f"{t['transform']:22s} {t['applied_fraction']:7.1%} "
          f"{t.get('n_altered', 0):>8,} {t['delta_macro_f1']:+10.4f} {cs}")
print()
print("Ranked by the conditional effect, which is what the rewrite costs on the")
print("files it actually touched. If that ordering differs from the aggregate")
print("ordering, the published ranking was an artefact of application rate and")
print("the paper's mechanism sentence has to be rewritten around this table.")

## 7. Save

In [ ]:
OUT = pathlib.Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)
reports = WORK / "aicd" / "eval" / "reports"
if reports.exists():
    shutil.copytree(reports, OUT / "reports", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/results", "zip", OUT)
print("-> /kaggle/working/results.zip")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.stat().st_size/1024:8.0f} KB  {p.relative_to(OUT)}")
elapsed("saved")